# Making SIMSOPT GPU native: safeguarded AL qualification

Select **Runtime > Change runtime type > GPU**, then run all cells. This workflow compares strict inner-stationarity and residual-like constraint mappings on `engineering`. It exports CPU/GPU VTS/VTU artifacts for every screen, and runs `stress` only if a candidate passes every scientific qualification gate. A result with no winner is a valid, actionable outcome.

In [ ]:
import subprocess
subprocess.run(["nvidia-smi"], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gpu-native-objective", "https://github.com/PedroFranciscoGil/simsopt.git", str(repo)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "pytest", "pyevtk"], cwd=repo, check=True)
os.chdir(repo)
source_root = repo / "src"
sys.path.insert(0, str(source_root))
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
print(f"Imported SIMSOPT from {resolved_package}")
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report
assert any(device.platform == "gpu" for device in jax.devices()), report

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/gpu"], cwd=repo, check=True)

In [ ]:
import shutil

artifact_root = Path("/content/simsopt-al-safeguards")
if artifact_root.exists():
    shutil.rmtree(artifact_root)
artifact_root.mkdir()
env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
subprocess.run([sys.executable, "benchmarks/gpu/sweep_augmented_lagrangian_safeguards.py", "--screen-problem", "engineering", "--final-problem", "stress", "--max-outer-iterations", "10", "--max-inner-iterations", "200", "--maxcor", "100", "--maxls", "50", "--current-scale", "100000", "--target-tile-size", "1024", "--source-tile-size", "4320", "--output-dir", str(artifact_root)], cwd=repo, env=env, check=True)

In [ ]:
import json

summary = json.loads((artifact_root / "safeguard-qualification-summary.json").read_text())
assert summary["schema_version"] == 1
assert summary["workflow"] == "augmented_lagrangian_safeguard_qualification"
assert len(summary["candidates"]) == 4
for candidate in summary["candidates"]:
    result = json.loads((artifact_root / candidate["result_file"]).read_text())
    assert result["schema_version"] == 7
    assert result["solver"]["require_inner_stationarity"]
    for backend in ("cpu", "gpu"):
        metrics = result[backend]["final_metrics"]
        assert {"objective", "normalized_normal_field", "coil_constraints"} <= metrics.keys()
        optimization = result[backend]["optimization"]
        assert {"final_constraints", "final_al_constraints", "terminated_by_inner_safeguard"} <= optimization.keys()
    for final_design in result["visualizations"].values():
        for artifact in final_design.values():
            path = Path(artifact)
            assert path.is_file() and path.stat().st_size > 0, path
if summary["production"]["executed"]:
    final_result = json.loads((artifact_root / summary["production"]["result_file"]).read_text())
    assert final_result["schema_version"] == 7
else:
    assert summary["winner"] is None
    assert summary["production"]["skip_reason"]
print(json.dumps({"qualified_indices": summary["qualified_indices"], "winner": summary["winner"], "production": summary["production"]}, indent=2))

## Interpretation

Qualification is binary and precedes ranking. The diagnostic order explains which rejected candidate came closest, but it cannot promote one. If no screen passes, production is deliberately skipped; inspect each CPU/GPU surface and coil export plus its inner-stage history before designing the next method.

In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/simsopt-al-safeguards", "zip", artifact_root)
files.download(archive)